# GPA Predictor — End-to-End Notebook

This notebook covers:
- Loading data
- Basic cleaning / checks
- Train/test split
- Model training (Linear Regression)
- Evaluation (MAE, RMSE, R²)
- Saving model to `models/gpa_model.pkl`

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

BASE_DIR = Path.cwd()
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/mock_gpa.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_PATH = MODEL_DIR / "gpa_model.pkl"

BASE_DIR, DATA_PATH, MODEL_PATH

(PosixPath('/content'),
 '/content/drive/MyDrive/Colab Notebooks/mock_gpa.csv',
 PosixPath('/content/models/gpa_model.pkl'))

## 1) Load data (CSV-only)

This notebook requires `data/mock_gpa.csv`. If the file is missing, it will stop with an error.

In [6]:
# if not DATA_PATH.exists():
#     raise FileNotFoundError(
#         f"Required dataset not found: {DATA_PATH}. "
#         "Please add data/mock_gpa.csv before running this notebook."
#     )

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/mock_gpa.csv")
print(f"Loaded dataset: {DATA_PATH}")
df.head()

Loaded dataset: /content/drive/MyDrive/Colab Notebooks/mock_gpa.csv


,study_hours,IQ,attendance,sleep_hours,GPA
0,33,93,67,6,3.154090
1,19,106,89,7,4.000000
2,12,88,50,4,2.412876
3,25,158,60,8,3.641077
4,23,94,77,8,3.788884


### Validate required columns

In [7]:
required_cols = {"study_hours", "IQ", "attendance", "sleep_hours", "GPA"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

# Keep only required columns (optional but helps avoid surprises)
df = df[["study_hours", "IQ", "attendance", "sleep_hours", "GPA"]].copy()
df.head()

,study_hours,IQ,attendance,sleep_hours,GPA
0,33,93,67,6,3.154090
1,19,106,89,7,4.000000
2,12,88,50,4,2.412876
3,25,158,60,8,3.641077
4,23,94,77,8,3.788884


## 2) Quick data checks

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   study_hours  100 non-null    int64  
 1   IQ           100 non-null    int64  
 2   attendance   100 non-null    int64  
 3   sleep_hours  100 non-null    int64  
 4   GPA          100 non-null    float64
dtypes: float64(1), int64(4)
memory usage: 4.0 KB


In [9]:
df.isna().sum()

,0
study_hours,0
IQ,0
attendance,0
sleep_hours,0
GPA,0


In [10]:
df.describe()

,study_hours,IQ,attendance,sleep_hours,GPA
count,100.000000,100.000000,100.000000,100.000000,100.000000
mean,21.450000,119.900000,75.900000,6.430000,3.618847
std,10.335166,23.654713,14.315704,1.701188,0.501122
min,5.000000,80.000000,50.000000,4.000000,2.269715
25%,12.000000,99.000000,65.750000,5.000000,3.329360
50%,21.500000,121.000000,77.000000,6.000000,3.911994
75%,29.000000,141.000000,86.250000,8.000000,4.000000
max,39.000000,159.000000,99.000000,9.000000,4.000000


## 3) Prepare features/target

Features:
- `study_hours`, `IQ`, `attendance`, `sleep_hours`

Target:
- `GPA`

In [11]:
X = df[["study_hours", "IQ", "attendance", "sleep_hours"]]
y = df["GPA"]

X.head(), y.head()

(   study_hours   IQ  attendance  sleep_hours
 0           33   93          67            6
 1           19  106          89            7
 2           12   88          50            4
 3           25  158          60            8
 4           23   94          77            8,
 0    3.154090
 1    4.000000
 2    2.412876
 3    3.641077
 4    3.788884
 Name: GPA, dtype: float64)

## 4) Train/test split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((80, 4), (20, 4))

## 5) Train model (Linear Regression)

In [13]:
model = LinearRegression()
model.fit(X_train, y_train)

model.coef_, model.intercept_

(array([0.01019417, 0.0073539 , 0.01751585, 0.0139277 ]),
 np.float64(1.0988527712644838))

## 6) Evaluate

In [15]:
y_pred = model.predict(X_test)
y_pred_clipped = np.clip(y_pred, 0, 4)

mae = mean_absolute_error(y_test, y_pred_clipped)
rmse = mean_squared_error(y_test, y_pred_clipped,)
r2 = r2_score(y_test, y_pred_clipped)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2:  {r2:.4f}")

MAE:  0.2868
RMSE: 0.1186
R^2:  0.6074


## 7) Save the trained model

This writes the model to `models/gpa_model.pkl` (the path used by your FastAPI app).

In [16]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)

print(f"Saved model to: {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size} bytes")

Saved model to: /content/models/gpa_model.pkl
File size: 905 bytes


## 8) Quick inference test (matches FastAPI input fields)

In [17]:
sample = {
    "study_hours": 20,
    "IQ": 110,
    "attendance": 85,
    "sleep_hours": 7,
}

x = np.array([
    sample["study_hours"],
    sample["IQ"],
    sample["attendance"],
    sample["sleep_hours"],
]).reshape(1, -1)

pred = float(np.clip(model.predict(x)[0], 0, 4))
pred

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


3.6980056722186854